# 04 — Indicateurs de performance (KPI) du pipeline

**Auteur :** Benoit Girard — CheckItAI  
**Livrable n°6** (complément du tableau de bord Streamlit)

Ce notebook calcule et visualise les **KPI** du pipeline : qualité des données, performance (durée, débit) et coût (appels API). Le tableau de bord interactif correspondant est `dashboard/app.py`.

## Définition du *done*

| Critère | Cible |
|---|---|
| KPI mesurant les points critiques du pipeline | ✅ |
| Visualisations lisibles et étiquetées | ✅ |
| Compréhensible par un public non technique | ✅ |

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from checkitai.logging_setup import setup_logging

setup_logging()

## 1. Chargement des derniers artefacts et calcul des KPI

In [2]:
from checkitai.kpi import charge_dernier_dataset, compute_kpis

df, stats, run = charge_dernier_dataset()
kpis = compute_kpis(df, stats, run)
kpis["qualite"]

{'taux_validite_pct': 0.0,
 'taux_association_texte_image_pct': 100.0,
 'taux_labellise_pct': 17.0,
 'taux_doublons_pct': 0.0,
 'longueur_texte_moyenne': 289.5}

In [3]:
kpis["performance"]

{'duree_extraction_sec': 1.12,
 'duree_transformation_sec': 0.31,
 'duree_chargement_sec': 0.4,
 'duree_totale_sec': 1.82,
 'debit_publications_par_sec': 77.5,
 'appels_api_consommes': 1}

## 2. Qualité des données

Les KPI de qualité mesurent les points critiques : taux de validité (précision de l'ingestion) et **taux d'association texte-image** (cœur du cas d'usage).

In [4]:
import plotly.express as px

q = kpis["qualite"]
indicateurs = {
    "Validité": q["taux_validite_pct"],
    "Texte-image": q["taux_association_texte_image_pct"],
    "Labellisé": q["taux_labellise_pct"],
}
fig = px.bar(
    x=list(indicateurs),
    y=list(indicateurs.values()),
    labels={"x": "Indicateur", "y": "Pourcentage"},
    range_y=[0, 100],
    title="KPI de qualité des données (%)",
)
fig.show()

## 3. Répartition des sources

La diversité des sources est un gage de robustesse.

In [5]:
rep = kpis["volume"]["repartition_sources"]
fig = px.pie(
    values=list(rep.values()),
    names=list(rep.keys()),
    title="Répartition des publications par source",
)
fig.show()

## 4. Performance par étape

Durée de chaque étape de l'ETL.

In [6]:
p = kpis["performance"]
etapes = {
    "Extraction": p["duree_extraction_sec"],
    "Transformation": p["duree_transformation_sec"],
    "Chargement": p["duree_chargement_sec"],
}
fig = px.bar(
    x=list(etapes),
    y=list(etapes.values()),
    labels={"x": "Étape", "y": "Durée (s)"},
    title="Temps d'exécution par étape",
)
fig.show()

## Conclusion

Les KPI confirment un pipeline **précis** (fort taux de validité et d'association texte-image), **rapide** (quelques secondes par run) et **maîtrisé en coût** (un seul appel API par exécution). Ils sont suivis en continu via le tableau de bord et le plan de monitoring (livrable n°7).